# UPI ANALYSIS

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np

In [ ]:
# Loading Dataset
df_upi = pd.read_csv(r'E:\Data Analystics\PROJECTS FOR PORTFOLIO\PROJECT 01_UPI-spending-analyzer\Data\upi_raw.csv')

print("Dataset loaded successfully")

In [ ]:
# Shape of the dataset
print("Number of Rows: ",df_upi.shape[0])
print("Number of Columns: ",df_upi.shape[1])

In [ ]:
# First look of the dataset
df_upi.head(10)

In [ ]:
# Column names and data types :
df_upi.info()

In [ ]:
# Basic Statistics
df_upi.describe()

In [ ]:
# Checking Missing Values
df_upi.isnull().sum()

In [ ]:
# Checking duplicates
print("Duplicated Rows: ",df_upi.duplicated().sum())

In [ ]:
# Unique Categories
print("Unique Categories: \n",df_upi["Category"].value_counts())

In [ ]:
# Unique Payment Modes
print("Unique Payment Modes: \n",df_upi["Payment_Mode"].value_counts())

#### Dataset Profiling Observations

| # | Question | Answer |
|---|---|---|
| 1 | Rows & Columns | 1020 rows, 8 columns |
| 2 | Date column data type | Object (str) - needs fixing |
| 3 | Missing values | Category: 30, Notes: 43 |
| 4 | Duplicate rows | 20 |
| 5 | Most transacted category | Food & Dining (168) |
| 6 | Most used payment mode | GPay (356 times) |

### Data Cleaning

In [ ]:
# Making copy of the dataframe
df_upi_copy = df_upi.copy()
print("Working Copy Created Successfully")

In [ ]:
# Cleaning Date Column : Changing Datatype from 'str' to 'datetime'
df_upi_copy["Date"] = pd.to_datetime(df_upi_copy["Date"])

# Checking
print("Data Type of Column Date: ",df_upi_copy["Date"].dtype)

print("Sample: \n",df_upi_copy["Date"].head(5))

In [ ]:
# Fixing missing notes with neutral placeholder
df_upi_copy["Notes"] = df_upi_copy["Notes"].fillna("No Description")

# Varifying
print("Missing in Notes column after fixing: ",df_upi_copy["Notes"].isnull().sum())

##### Fixing Missing Category Column for Merchants

In [ ]:
# We will not drop the rows, instead we will fill the missing categories by the use of merchant column.

# Finding Unique Merchants with Missing Category
df_upi_copy[df_upi_copy["Category"].isnull()]["Merchant"].unique()


In [ ]:
# Merchants with Missing Categories
df_upi[df_upi["Category"].isnull()][["Merchant", "Category"]].drop_duplicates()

In [ ]:
# Building a merchant-to-category reference dictionary for merchant with missing category

merchant_category_map = {
    "Zomato" : "Food & Dining", "Amazon" : "Shopping", "Swiggy" : "Food & Dining", "Local Dhaba" : "Food & Dining", "BookMyShow" : "Entertainment", "BigBasket" : "Groceries", "Dominos" : "Food & Dining", "Cafe Coffee Day" : "Food & Dining", "Uber" : "Transport", "PVR Cinemas" : "Entertainment", "YouTube Premium": "Entertainment", "upGrad" : "Education", "Tata Sky":"Entertainment", "Rapido" : "Transport", "McDonald's" : "Food & Dining", "ATM Withdrawal" : "Miscellaneous", "DMart" : "Groceries", "Gift" : "Miscellaneous", "Goibibo" : "Travel", "Myntra" : "Shopping", "Ajio" : "Shopping", "Local Kirana" : "Groceries", "Spotify" : "Subscriptions", "Unacademy" : "Education", "Netflix" : "Subscriptions"
}

In [ ]:
# Fixing missing Category using Merchant name

df_upi_copy["Category"] = df_upi_copy.apply(
    lambda row: merchant_category_map.get(row["Merchant"], row["Category"])
    if pd.isnull(row["Category"]) else row["Category"],
    axis=1
)

# Check how many are still missing after smart fill
print("Missing in Category after fixing:", df_upi_copy["Category"].isnull().sum())

#####  Fixing missing Type column for Category

In [ ]:
# Checked missing type columns, there are no missing types in that column. No need to fix this.

#### Removing Duplicates

In [ ]:
# Checking Duplicate Rows
print("Number of Duplicate Rows Before Removing: ",df_upi_copy.duplicated().sum())

In [ ]:
# Removing Duplicates
df_upi_copy = df_upi_copy.drop_duplicates(keep="first")

# Checking after removing duplicates
print("Number of Duplicate Rows after Removing: ",df_upi_copy.duplicated().sum())

In [ ]:
# Final Missing Value Check
print("Remaining Missing Values after Cleaning: \n",df_upi_copy.isnull().sum())

In [ ]:
# Cleaning Summary
print("========Cleaning Summary========")
print(f"Total Rows after cleaning: {len(df_upi_copy)}")
print(f"Remaining Missing Values: {df_upi_copy.isnull().sum().sum()}")
print(f"Data Type of Date Column: ",df_upi_copy["Date"].dtype)
print(f"Categories: {df_upi_copy["Category"].nunique()}")
print(f"Date Range: {df_upi_copy["Date"].min()} to {df_upi_copy["Date"].max()}")

#### Feature Engineering

In [ ]:
# Extracting date features from Date column
df_upi_copy["Month_Num"] = df_upi_copy["Date"].dt.month
df_upi_copy["Month_Name"] = df_upi_copy["Date"].dt.month_name()
df_upi_copy["Day_of_Week"] = df_upi_copy["Date"].dt.strftime("%A")
df_upi_copy["Week_Number"] = df_upi_copy["Date"].dt.isocalendar().week

df_upi_copy.head(5)

In [ ]:
# Creating Spend Bucket column
def categorize_spend(amount):
    if amount < 500:
        return "Small"
    elif amount <= 2000:
        return "Medium"
    else:
        return "Large"
    
df_upi_copy["Spend_Bucket"] = df_upi_copy["Amount"].apply(categorize_spend)

# Verifying
print(df_upi_copy["Spend_Bucket"].value_counts())

In [ ]:
# Creating Is_Weekend column
def weekend_flag(day_of_week):
    if day_of_week == "Saturday":
        return "Yes"
    elif day_of_week == "Sunday":
        return "Yes"
    else:
        return "No"
    
df_upi_copy["Is_Weekend"] = df_upi_copy["Day_of_Week"].apply(weekend_flag)

# Verifying
print(df_upi_copy["Is_Weekend"].value_counts())

In [ ]:
# Flagging transactions that are unusually high using the statistical method:

# Calculating Mean and Standard Deviation of Amount
mean_amount = df_upi_copy["Amount"].mean()
std_amount = df_upi_copy["Amount"].std()

# Anomaly Threshold
threshold = mean_amount + (2 * std_amount)

print(f"Mean of Amount: ₹{mean_amount.round(2)}\n"
      f"Standard Deviation of Amount: ₹{std_amount.round(2)}\n"
      f"Anomaly Threshold: ₹{threshold.round(2)}")

# Flagging Anomalies
def anomalies(amt):
    if amt > threshold:
        return "Yes"
    else:
        return "No"
    
df_upi_copy["Is_Anomaly"] = df_upi_copy["Amount"].apply(anomalies)

# Verifying
print(df_upi_copy["Is_Anomaly"].value_counts())

### 📌 Anomaly Threshold
- Threshold: ₹6,603
- Flagged: 35 anomalies out of 1000 transactions (3.5%)
- Detailed interpretation covered in Analysis phase.

In [ ]:
# Creating Quarter Column:
df_upi_copy["Quarter"] = df_upi_copy["Date"].dt.quarter.map({
    1 : "Q1",
    2 : "Q2",
    3 : "Q3",
    4 : "Q4"
})

# Verifying
print(df_upi_copy["Quarter"].value_counts().sort_index())

In [ ]:
# Checking new shape and all columns

print("Final Shape after cleaning: ",df_upi_copy.shape)
print("\nAll Columns: \n",df_upi_copy.columns.to_list())
print("\nSample of Dataset: \n")
df_upi_copy.head(10)

In [ ]:
# Saving Cleaned File
df_upi_copy.to_csv(r"E:\Data Analystics\PROJECTS FOR PORTFOLIO\PROJECT 01_UPI-spending-analyzer\Data\upi_cleaned.csv", index=False)

print("Cleaned File Saved Successfully.")